# Module 3 — NGS Pipeline with Claude Code (Alignment)

**SBML Lab Intern Training | KAIST**

---

This module covers the full NGS alignment pipeline — from raw SRA data to a sorted, indexed BAM, a GFF file, and a visualization of the ChIP-exo signal in MetaScope. You'll use Claude Code to generate the pipeline commands, but you must understand what each step does before running it.

**Lab context:** Reference genome is *E. coli* K-12 MG1655. The dataset is **single-end ChIP-exo reads** — each read comes from one end of the DNA fragment, not a pair. Aligner is `bowtie2`. Post-alignment processing uses `samtools`, followed by `makegff.py` to produce a GFF file for MetaScope visualization.

**Learning objectives:**
- Understand how `bowtie2` aligns short reads to a reference genome
- Understand why BAM files must be sorted and indexed before downstream use
- Practice using Claude Code's **plan mode** for multi-step pipelines
- Annotate and verify every flag in a generated command before running it

## Background: NGS and ChIP-exo

### What is next-generation sequencing (NGS)?

A sequencing machine reads millions of short DNA fragments (called **reads**) in parallel and reports their sequences along with quality scores for each base. A single ChIP-exo experiment produces 10–50 million reads, each 50–150 base pairs long. Raw reads are stored in **FASTQ** format — one file, millions of entries.

Reads alone don't tell you anything. You need to figure out *where* in the genome each read came from. That's what alignment does.

### What is ChIP-exo?

**ChIP-exo** is a technique that maps exactly where a DNA-binding protein (a transcription factor) attaches to the chromosome:

1. **ChIP (chromatin immunoprecipitation):** Protein-DNA complexes are chemically cross-linked and then broken into fragments. An antibody specific to your protein of interest pulls down only the fragments it was bound to.
2. **Exo (exonuclease treatment):** A DNA-chewing enzyme trims the pulled-down fragments from their ends, leaving only the sequence right at the binding boundary. This is what makes ChIP-*exo* more precise than regular ChIP-seq.
3. **Sequencing:** The remaining fragments are sequenced. The resulting reads cluster tightly around the exact binding site.

### Why does this lab do ChIP-exo?

This lab studies transcription factors (TFs) — proteins that bind specific DNA sequences to control nearby genes. ChIP-exo lets us answer: *"Where in the E. coli genome does TF X bind, and under what conditions?"*

The dataset you'll work with in Modules 3 and 4 is a FUR ChIP-exo experiment — mapping where the iron-sensing protein FUR binds across the entire *E. coli* chromosome.

### The pipeline in one sentence

`FASTQ reads` → **align to reference genome** → `SAM/BAM alignment file` → **sort and index** → **convert to GFF** → `visualize in MetaScope`

Each step in this pipeline is one section of this module.

---

## 0. File Formats — Know Before You Align

This pipeline produces and consumes several file formats. Use Claude Code to answer the questions below **before** starting the exercises.

| Format | Question to answer with Claude Code |
|--------|-------------------------------------|
| FASTQ  | What do the 4 lines per read represent? What does the quality score number mean? |
| FASTA  | How does it differ from FASTQ? When would you use one vs the other? |
| SAM    | How many columns are there? What does the FLAG field encode? |
| BAM    | Why is BAM preferred over SAM for storage and downstream tools? |
| GFF    | How many columns? What coordinate system (0-based or 1-based)? |

Write your answers below — one sentence per format.


> **Answer**
>
> FASTQ : sequencing read 하나를 4줄로 표현하는 포맷, 서열 정보와 신뢰도를 저장. (1:read ID / 2:실제 염기 서열 / 3:구분자 / 4:quality score, Q=-10(log(P)),이때 P는 해당 염기가 틀릴 확률)
>
> FASTA : FASTQ와 달리 sequence info만 담는 포맷, FASTA는 Reference genome이나 특정 유전자 서열을 추출해서 볼 때, FASTQ는 실험의 원본 데이터를 이용할 때 사용.
>
> SAM : 결과 담는 포맷, 11개의 필수 컬럼이 존재, FLAG field에는 이진수를 이용해서 각 자릿수에 정보를 저장. 1번째 자릿수에 paired-end read, 2번째 자릿수에 정렬이 정상적으로 이뤄졌는지 등
>
> BAM : 정보를 이진 파일로 저장하여 SAM에 비해 작고 처리가 빠름. bowtie2를 통해 처리 한 작업을 저장/처리할 때 사용하는 포맷
>
> GFF : genome상의 feature의 위치 및 속성을 기록하는 포맷, coordinate system은 1-based로, start=1은 서열의 첫번째 염기를 의미함.

## 1. How `bowtie2` Works

### What is alignment?

**Alignment** is the process of finding where each sequenced read comes from in the reference genome. Imagine you have 20 million short sentences (reads), each 75 words long, and you want to find where each one appears in a 4.6-million-word book (the *E. coli* genome). Alignment does that — at scale, in seconds.

The output is a file where every read is annotated with its genomic position: chromosome, start coordinate, strand, and a mapping quality score.

### Why do we need an index?

Searching the entire genome for every read would be impossibly slow. `bowtie2-build` pre-processes the reference genome into a compressed **index** — a data structure that allows near-instant lookup. Think of it like a book index: instead of reading every page to find "FUR binding site," you jump straight to the relevant pages.

You build the index once; then alignment against that genome uses it every time.

`bowtie2` aligns short reads to a reference genome using an index built with `bowtie2-build`. Use Claude Code to understand how the aligner works before you run it.

### Exercise 1 — Understand bowtie2 Before Running It

Use Claude Code to understand how `bowtie2` finds alignments and what the key flags control. Write a 3-sentence summary in your own words in the cell below.

> **Answer**
>
> bowtie2는 시퀀싱 이후 만들어진 short read가 전체 유전체 중 어느 위치에 들어가는지 탐색하는 방법이다. 하지만 시퀀싱 후 수백만개의 read가 생성되고 이를 전부 alignment하게 되면 처리량이 현저히 많아지게 된다. 이때, bowtie2-build를 이용하여 reference genone에 index를 설정하면, bowtie2를 통해 유전체 위치 탐색이 용이하게 된다.

## 2. `samtools` Internals

### Why sort a BAM file?

After alignment, reads are stored in the order they were sequenced — essentially random with respect to genomic position. Most downstream tools (genome browsers, variant callers, coverage calculators) need to jump to a specific genomic region quickly. A **coordinate-sorted BAM** organizes reads by position, so a tool looking at position 1,234,567 doesn't have to scan the whole file — it jumps directly there.

**You must sort before you index.** An index built on an unsorted file is meaningless because the file's structure doesn't match what the index promises.

### Why index a BAM file?

The `.bai` index file is a lookup table that maps genomic positions to byte offsets in the BAM file. It's what allows `samtools view` to extract reads from chromosome position X in milliseconds. Without it, every query would require reading the entire BAM from the start.

`samtools` converts, sorts, and indexes alignment files. Use Claude Code to understand the difference between SAM and BAM, and why the sort step must come before the index step.

### Exercise 2 — Sort Before Index

**Without asking Claude Code first:** write in the cell below why you must sort a BAM file before indexing it. Use your own reasoning.

Then verify your explanation with Claude Code.

> **Answer**
>
> 색인을 만드는 이유는 탐색 및 정보 접근에 용이하기 위해서이다. read는 만들어지는 순서대로 bowtie2를 통해 BAM 파일로 저장되므로, sorting을 하지 않고 색인을 만든다면 무분별한 순서대로 색인이 만들어진다. index를 만드는 목적을 생각하여 위치별로 순서화 한 후 색인을 만드는 것이 적절하다.

## 3. Plan Mode — Reviewing Pipelines Before Running

Claude Code's **plan mode** lets you see every step Claude intends to take before a single command is executed.

**How to activate:** Press **Shift+Tab** before sending your prompt, cycling until the status line shows plan mode is on (a single press may land on a different mode). The interface switches to plan mode. Claude will show you the full sequence of actions — file reads, commands, edits — and wait for your approval.

**Why this matters for pipelines:**  
A multi-step NGS pipeline touches raw data, builds index files, and writes intermediate outputs. A mistake at step 2 (wrong index path) silently produces a valid-looking but incorrect BAM. Reviewing the plan lets you:
- Catch wrong paths or flag values before they cause silent errors
- Understand every command before it runs
- Modify individual steps (e.g., add `--no-unal`, increase `-p` threads) without re-generating the whole plan

**Rule for this module:** Any time you ask Claude Code to generate a multi-step pipeline, you must use plan mode and review it before approving.

### Exercise 3 — Plan Mode Pipeline Review

Use plan mode (Shift+Tab twice) to generate the full pipeline:
```
FASTQ → bowtie2 align → samtools sort → samtools index → makegff.py
```
Review every proposed step before approving. Identify at least one flag or parameter you would change or verify, and explain why.

> **답변**
>
> 확인 및 수정한 부분은 bowtie2 align 부분의 --no unal 부분. --no unal의 경우, reference genme에서 정렬되지 않은 read를 SAM파일에 기록하지 않는데, 이럴 경우, 전체 read의 수를 확인할 수 없어 필요시 정보 확인이 불가능해짐. -> --no unal은 유지하되, 별도의 로그 파일에 저장하도록 변경
>
> full pipeline 작성 요구 시, makegff.py가 만드는 GFF는 reference FASTA의 accession(NC_000913.3)을 그대로 header로 쓰게 되는데, read 쪽 GFF(makegff.py 출력)와 reference 쪽 GFF(annotation)의 header가 서로 다르면 MetaScope에서 두 파일이 겹쳐지지 않으므로, 두 파일의 header가 동일해지도록 설정

## 4. Downloading Your Input Data

The pipeline needs **two** inputs, both in `data/reference/`: the sequencing **reads** and the **reference genome** they align to. You'll download both with Claude Code in this section.

### Exercise 4 — Reads: single-end ChIP-exo (`fastq-dump`)

The reads come from the FUR ChIP-exo study you'll work with in Module 4: Seo et al. 2014, GEO series **GSE54901**. Use Claude Code to find and download an appropriate single-end ChIP-exo sample into `data/reference/`.

In [ ]:
%%bash
# Exercise 4 — Use Claude Code to download the ChIP-exo reads from GSE54901.
# Paste the final command(s) here and annotate each flag with a comment.
# Note: this notebook's shell cwd is notebooks/, so repo-root paths need a ../ prefix.

prefetch SRR1168121 -O ../data/reference/
# prefetch: SRA 서버에서 SRR1168121의 원본 .sra 파일을 받음
# -O ../data/reference/ : 다운로드 결과를 data/reference/SRR1168121/SRR1168121.sra 경로에 저장

fastq-dump --gzip -O ../data/reference/ ../data/reference/SRR1168121/SRR1168121.sra
# fastq-dump: .sra를 FASTQ로 변환. single-end read이므로 --split-files는 사용하지 않음
#             (paired-end일 때만 R1/R2 두 파일로 분리해야 함).
# --gzip     : 결과 FASTQ를 압축 저장. bowtie2 등 대부분의 툴이 .fastq.gz를 그대로 입력받을 수 있어 용량 절약 가능
# -O ../data/reference/ : 결과 SRR1168121.fastq.gz를 이 디렉토리에 저장

rm -rf ../data/reference/SRR1168121
# prefetch가 만든 중간 .sra 캐시 폴더는 fastq 변환 후 더 이상 필요 없으므로 정리

ls -lh ../data/reference/SRR1168121.fastq.gz


### Exercise 4b — Reference Genome: *E. coli* K-12 MG1655 (`NC_000913.3`)

Alignment needs the genome the reads came from. Use Claude Code to download the *E. coli* K-12 MG1655 reference genome — accession **NC_000913.3** — as a FASTA into `data/reference/`, saved as `NC_000913.3.fasta`. (Hint: the `entrez-direct` tools in this container can fetch sequences by accession from NCBI.)

This is the file `bowtie2-build` turns into the alignment index in Exercise 7 — without it the pipeline's first step fails.

In [4]:
%%bash
# Exercise 4b — Use Claude Code to download the reference genome NC_000913.3 as FASTA
#               into data/reference/NC_000913.3.fasta.
# Note: this notebook's shell cwd is notebooks/, so repo-root paths need a ../ prefix.

efetch -db nuccore -id NC_000913.3 -format fasta > ../data/reference/NC_000913.3.fasta
ls -lh ../data/reference/NC_000913.3.fasta

# efetch(entrez-direct)는 NCBI nuccore 데이터베이스에서 accession(NC_000913.3)으로
# 서열을 직접 조회해서 FASTA로 가져온다. 웹사이트를 탐색할 필요 없이 accession만 알면
# 바로 받을 수 있어 재현 가능하고 안정적이다.

curl: (56) OpenSSL SSL_read: OpenSSL/3.5.6: error:0A000126:SSL routines::unexpected eof while reading, errno 0
 ERROR:  curl command failed ( Thu Jul 16 05:37:19 UTC 2026 ) with: 56
-X POST https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi -d db=nuccore&id=NC_000913.3&rettype=fasta&retmode=text&tool=edirect&edirect=25.3&edirect_os=Linux&email=vscode%40codespaces-b37f35
HTTP/1.0 200 OK


-rw-rw-rw- 1 vscode vscode 4.5M Jul 16 05:37 ../data/reference/NC_000913.3.fasta


## 5. SAM FLAG Values

Every SAM record has a FLAG field in column 2 — a bitwise integer encoding alignment properties. Use Claude Code to understand what FLAG values mean and how to filter by them with `samtools view`.

### Exercise 5 — Decode an Unknown FLAG

> Look up what a specific FLAG value means using Claude Code. Choose a value you have **not** looked up yet (e.g., 2048, 256, 1024).
>
> Write:
> 1. The FLAG value you chose
> 2. What it means (decoded bits)
> 3. How you would verify that a real SAM record carries this flag (e.g., with `samtools view -f` or `samtools flags`)

> **Answer**
>
> 1. 16
> 2. read가 reference genome의 reverse strand에 정렬됨을 의미.
> 3. samtools view -f 16 SRR1168121.sam | wc -l -> 실행으로 1,395,830건의 flag 확인 가능

## 6. Writing `makegff.py`

Your sorted BAM holds every aligned read in a binary format bioinformatics tools understand — but **MetaScope**, the lab's genome browser, reads **GFF**, not BAM. `makegff.py` is the converter, and it's the last step of the pipeline.

But it isn't a dumb format swap — *how* you turn reads into GFF rows decides whether you can actually see a binding site. Two things drive that:

**MetaScope draws each GFF feature at a height set by the score column.** So the score is not decoration — it's the signal. If you want a binding site to rise as a **peak**, the score has to carry *how much evidence* is at each position.

**ChIP-exo puts the signal at the read's 5′ end.** A lambda exonuclease chews each bound DNA fragment 5′→3′ until it stops at the protein–DNA crosslink. So the **5′ end of every read marks a crosslink boundary** — that's the informative position, not the whole read body:
- for a **`+`-strand** read, the 5′ end is its **start**;
- for a **`−`-strand** read, the 5′ end is its **end**.

Many fragments from the same binding event pile their 5′ ends at the same spot. So the right GFF encodes, **per strand**, *how many reads share each 5′-end position* — and writes that **count into the score column**. That's what makes a binding site show up as a tall, sharp peak. (The crosslink is flanked by a `+`-strand peak and a `−`-strand peak; the true site sits **between** them — which is why strands are kept separate.)

This script is **not pre-provided** — you'll write it with Claude Code. Consult `lab-context.md` for the exact GFF column structure this lab uses.

### Exercise 6 — Write `makegff.py` (5′-end pileup)

Use Claude Code to write a Python script `makegff.py` that turns a sorted BAM into a GFF MetaScope can display as a ChIP-exo signal:

1. Takes a sorted BAM as input, writes a GFF file.
2. For each aligned read, take its **5′ end** (`start` if `+`, `end` if `−`).
3. **Count reads per (strand, 5′-end position)** and write **one GFF row per occupied position**, with a **score derived from that count** so pile-ups become peaks.
4. Keep `+` and `−` strands **separate** (this is the lab script's `--separate_strand` behavior).
5. Normalize the score so a few very strong sites don't visually flatten the weaker real ones (see below).

**Output format** (one row per occupied 5′-end position; coordinates **1-based**):
```
NC_000913.3	makegff	fiveprime	12345	12345	7	+	.	depth=12345
```
Columns: seqname, source (`makegff`), feature (`fiveprime`), start, end (= same position), **score**, strand, `.`, attribute (`depth=` always keeps the raw read count regardless of the score's scale).

Test it on your aligned BAM before running the full pipeline in Exercise 7.

> **Turned out to matter in practice:** on the real dataset, raw counts range from 1 to several thousand (with rare artifact positions far higher). We tried a few scoring schemes:
> - **raw count as score** — a handful of extreme positions dominate MetaScope's height axis; every weaker (but still real) peak looks flat.
> - **`log2(count+1)`** — still unbounded, so an artifact position (e.g. one that pushed the axis out to ~24,000,000) can still blow out the scale and hide every real peak.
> - **normalize this run's own counts onto a fixed `1..10` range (`compute_score`, signed by strand → effectively `-10..10`)** — this is the version that actually showed visible peaks in MetaScope, because the tallest peak is *always* exactly 10 no matter how extreme the raw counts get, so weaker real peaks stay visible relative to it. This is the version implemented below and used in Exercise 7.

> **Explain it:** after your code runs, write 1–2 sentences on *why counting 5′ ends per strand (rather than one row per read) is what makes a binding site visible*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

> **답변**
>
> makegff.py는 정렬된 BAM을 pysam으로 읽어, 각 read의 5′ end 위치(+가닥은 정렬 시작 위치, −가닥은 정렬 끝 위치)를 구한다. 몇 개의 read가 그 위치에 겹치는지 카운트해서, 실제로 read가 존재하는 위치마다 딱 한 행만 기록하고 그 카운트를 score의 기반으로 삼는다. 결합부위에서는 exonuclease가 단백질에 막혀 멈추는 지점에 여러 read의 5′ end가 몰리므로 count가 크게 쌓여 뾰족한 피크로 보이고, 무작위 구간은 5′ end가 여기저기 흩어져 있어 count가 1~2가 되므로 평평하게 보인다.
>
> run의 최댓값을 기준으로 count를 1~10 범위로 정규화하고, strand는 부호로 구분하는 방식을 사용하였다.


In [8]:
%%writefile makegff.py
#!/usr/bin/env python3
import math
import sys
from collections import Counter

import pysam

TARGET_MAX_SCORE = 10


# Pileup counts span a huge range (1 to several thousand, and occasional
# artifact positions much higher), so a raw or log2(count+1) score still lets
# a handful of outlier positions dominate the GFF score axis and flatten every
# other real peak near zero. Normalizing this run's own counts onto a fixed
# 1..TARGET_MAX_SCORE range guarantees the tallest peak is always exactly
# TARGET_MAX_SCORE, so weaker (but still real) peaks stay visible.
def compute_score(count, max_count, target_max=TARGET_MAX_SCORE):
    if max_count <= 1:
        return 1
    score = 1 + (target_max - 1) * math.log(count) / math.log(max_count)
    return round(score)


def main():
    if len(sys.argv) != 3:
        sys.exit(f"Usage: python {sys.argv[0]} <sorted.bam> <output.gff>")
    bam_path, gff_path = sys.argv[1], sys.argv[2]

    # counts[chrom][(pos, strand)] = 해당 (위치, strand)에 5' end가 몰린 read 개수
    counts = {}

    with pysam.AlignmentFile(bam_path, "rb") as bam:
        for read in bam.fetch(until_eof=True):
            if read.is_unmapped:
                continue
            strand = "-" if read.is_reverse else "+"
            # ChIP-exo 시그널은 read의 5' end: +가닥은 시작, -가닥은 끝
            pos = read.reference_start + 1 if strand == "+" else read.reference_end
            # FASTA accession 버전(.3)을 떼어 lab annotation의 chromosome ID(NC_000913)와 맞춤
            chrom = read.reference_name.split(".")[0]
            counts.setdefault(chrom, Counter())[(pos, strand)] += 1

    max_count = max(
        (c for positions in counts.values() for c in positions.values()), default=1
    )

    with open(gff_path, "w") as gff:
        # (pos, strand) 순으로 정렬 -> 파일 전체가 좌표 기준으로 단조 증가하는
        # coordinate-sorted GFF가 된다 (strand별로 통째로 묶어서 쓰면 좌표가
        # 중간에 처음으로 되돌아가서, 좌표 정렬을 기대하는 뷰어에서 뒤쪽 strand의
        # 뒷부분 데이터가 로드되지 않는 문제가 있었다).
        for chrom, positions in sorted(counts.items()):
            for (pos, strand), count in sorted(positions.items()):
                score = compute_score(count, max_count)
                # 부호로 strand를 한 번 더 표시(컬럼7의 +/-와 중복되지만, MetaScope에서
                # 두 strand 트랙을 score 축 위아래로 갈라 보기 위해 유지한다).
                if strand == "-":
                    score = -score
                gff.write(
                    f"{chrom}\tmakegff\tfiveprime\t{pos}\t{pos}\t{score}\t{strand}\t.\tdepth={count}\n"
                )


if __name__ == "__main__":
    main()


Overwriting makegff.py


## 7. Running the Full Pipeline

### Exercise 7 — Run the Full Pipeline

By now `data/reference/` should hold both inputs from Section 4: the reference genome (`NC_000913.3.fasta`, Exercise 4b) and the ChIP-exo reads (Exercise 4). Use plan mode to generate the full pipeline, then fill in each command below after reviewing the plan.

> **Explain it:** after your code runs, write 1–2 sentences on *how it works and why Claude chose that approach*. You should be able to defend every line — if you can't explain it, you don't yet understand it.

In [10]:
%%bash
# Full NGS pipeline
# Note: this notebook's shell cwd is notebooks/, so repo-root paths need a ../ prefix.

# Step 1: Build bowtie2 index from reference FASTA
bowtie2-build ../data/reference/NC_000913.3.fasta ../data/reference/NC_000913.3
# fasta 포맷을 통해 referenge genome을 bowtie2 index로 변환

# Step 2: Align single-end reads with bowtie2
bowtie2 -x ../data/reference/NC_000913.3 -U ../data/reference/SRR1168121.fastq.gz -S ../SRR1168121.sam
# fastq 포맷을 통해 read를 reference genome에 정렬하고 SAM 포맷으로 저장

# Step 3: Convert SAM to BAM (samtools view)
samtools view -bS ../SRR1168121.sam > ../SRR1168121.bam
# samtools view로 SAM 포맷을 BAM 포맷으로 변환

# Step 4: Sort BAM by coordinate (samtools sort)
samtools sort ../SRR1168121.bam -o ../SRR1168121_sorted.bam
# samtools sort로 BAM 파일을 좌표 기준으로 정렬

# Step 5: Index sorted BAM (samtools index)
samtools index ../SRR1168121_sorted.bam
# samtools index로 정렬된 BAM 파일에 대한 인덱스 생성

# Step 6: Run makegff.py to generate GFF for MetaScope
python makegff.py ../SRR1168121_sorted.bam chipexo.gff
#(strand, 5' end 위치)별로 겹치는 read 수를 세어 그 위치 하나에
# count를 score로 기록하므로, 같은 지점에 몰린 read들이 GFF 상에서 뾰족한 peak으로 나타남
# run의 최댓값 기준으로 count를 1~10 범위로 정규화하고(compute_score) strand는 부호로 구분하는 방식으로 진행

Settings:
  Output files: "../data/reference/NC_000913.3.*.bt2"
  Line rate: 6 (line is 64 bytes)
  Lines per side: 1 (side is 64 bytes)
  Offset rate: 4 (one in 16)
  FTable chars: 10
  Strings: unpacked
  Max bucket size: default
  Max bucket size, sqrt multiplier: default
  Max bucket size, len divisor: 4
  Difference-cover sample period: 1024
  Endianness: little
  Actual local endianness: little
  Sanity checking: disabled
  Assertions: disabled
  Random seed: 0
  Sizeofs: void*:8, int:4, long:8, size_t:8
Input files DNA, FASTA:
  ../data/reference/NC_000913.3.fasta


Building a SMALL index


Reading reference sizes
  Time reading reference sizes: 00:00:00
Calculating joined length
Writing header
Reserving space for joined string
Joining reference sequences
  Time to join reference sequences: 00:00:00
bmax according to bmaxDivN setting: 1160413
Using parameters --bmax 870310 --dcv 1024
  Doing ahead-of-time memory usage test
  Passed!  Constructing with these parameters: --bmax 870310 --dcv 1024
Constructing suffix-array element generator
Building DifferenceCoverSample
  Building sPrime
  Building sPrimeOrder
  V-Sorting samples
  V-Sorting samples time: 00:00:00
  Allocating rank array
  Ranking v-sort output
  Ranking v-sort output time: 00:00:00
  Invoking Larsson-Sadakane on ranks
  Invoking Larsson-Sadakane on ranks time: 00:00:00
  Sanity-checking and returning
Building samples
Reserving space for 12 sample suffixes
Generating random suffixes
QSorting 12 sample offsets, eliminating duplicates
QSorting sample offsets, eliminating duplicates time: 00:00:00
Multikey QSor

Renaming ../data/reference/NC_000913.3.3.bt2.tmp to ../data/reference/NC_000913.3.3.bt2
Renaming ../data/reference/NC_000913.3.4.bt2.tmp to ../data/reference/NC_000913.3.4.bt2
Renaming ../data/reference/NC_000913.3.1.bt2.tmp to ../data/reference/NC_000913.3.1.bt2
Renaming ../data/reference/NC_000913.3.2.bt2.tmp to ../data/reference/NC_000913.3.2.bt2
Renaming ../data/reference/NC_000913.3.rev.1.bt2.tmp to ../data/reference/NC_000913.3.rev.1.bt2
Renaming ../data/reference/NC_000913.3.rev.2.bt2.tmp to ../data/reference/NC_000913.3.rev.2.bt2
[WARNING] Failed to launch x86-64-v3 version, staying with default
[WARNING] Failed to launch x86-64-v3 version, staying with default
2631506 reads; of these:
  2631506 (100.00%) were unpaired; of these:
    20310 (0.77%) aligned 0 times
    2533623 (96.28%) aligned exactly 1 time
    77573 (2.95%) aligned >1 times
99.23% overall alignment rate


## 8. Visualizing ChIP-exo in MetaScope

You now have a GFF where each 5′-end pile-up is scored by read count. Numbers in a file don't tell you where FUR binds — you need to *see* the peaks along the genome next to the genes they sit near. That's what a **genome browser** does.

### What is a genome browser?

A genome browser draws the genome as a horizontal axis (position 1 → 4.6 Mb for *E. coli*) and stacks your data on top as **tracks**. Each track is one GFF file. Load the gene annotation as one track and your ChIP-exo pile-up as another, and you can read directly off the screen: *this peak sits right in front of that gene.*

### MetaScope — the lab's browser

**MetaScope** is the SBML lab's own genome browser. It reads **GFF files** (exactly what `makegff.py` produced), drawing each feature at a height set by its **score**, and lets you overlay tracks, zoom, jump to a position, search by gene name, and export a publication-quality figure.

- **It is a desktop application** — it runs on your own computer (Windows or macOS), **not** inside the Codespace. The Codespace is where you *make* the GFF; MetaScope is where you *look* at it.
- **Download and install it from the lab homepage** ([sbml-lab.ai](https://sbml-lab.ai)). Pick the build for your operating system.
- The workflow is: **produce the GFF in the Codespace → download the GFF to your machine → open it in MetaScope → cross-reference with the annotation → export a figure.**

### The two tracks you'll load

| Track | File | What it shows |
|-------|------|---------------|
| Gene annotation | `ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (from Module 2) | Where every gene is |
| ChIP-exo 5′-end pile-up | your `makegff.py` output (Exercise 6/7) | Read-count peaks where FUR-bound fragments piled up |

Overlaid, a tall peak of ChIP-exo 5′ ends just upstream of a gene is a candidate **FUR binding site** regulating that gene.

### Exercise 8 — What Should a Binding Site Look Like? (write before you look)

**Before installing anything**, write your prediction in the cell below — reason it out yourself:

1. ChIP-exo enriches DNA fragments that were bound by the FUR protein. When those reads are aligned and drawn as a track, **what shape** would you expect at a real binding site versus a random stretch of genome?
2. FUR is a **repressor** of iron-uptake genes. Relative to a gene it controls, **where** along the genome would you expect its binding site to sit — inside the gene, or somewhere else?

Then verify your reasoning with Claude Code (`/explain ChIP-exo peak shape` or ask directly).

> **Answer (instructor key)**
>
> 1 : 무작위 구간에서는 read가 거의 없거나 있어도 배경 수준으로 낮고 평평하게 퍼져 있을 것이다. 반면 실제 FUR 결합 부위에서는 exonuclease가 단백질에 막혀 멈추는 지점까지 DNA를 깎아내므로, 5' end가 결합 경계에 몰려 쌓인다. makegff.py가 strand별로 5' end만 카운트하므로, 하나의 정규분포형 봉우리가 아니라 +가닥과 -가닥에서 각각 좁고 뾰족한 봉우리가 결합 부위 양쪽에 나타나는 형태일 것이고, 실제 crosslink 지점은 그 두 봉우리 사이에 위치할 것이다.
>
> 2 : FUR은 세포 내 철의 농도에 따라 결합 활성이나 철 흡수 관련 인자의 전사를 조절하는 repressor로, binding site는 gene의 operator 부근일 것이다. 즉, 전사 시작점의 upstream 또는 그 부근에 결합하여 전사 개시를 방해할 것이다.

### Exercise 9 — Get Your GFFs Out of the Codespace

MetaScope runs on your machine, so both tracks need to be on your machine. Run the cell below to confirm both GFFs exist and see their sizes, then download them.

**To download a file from a Codespace:** in the VS Code file explorer (left panel), right-click the file → **Download…**, and save it somewhere you'll find it (e.g. your Desktop).

Download **both**:
- `data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff` (the annotation)
- your ChIP-exo GFF from Exercise 6/7 (e.g. `notebooks/chipexo.gff` — use whatever name you gave it)

In [9]:
%%bash
# Confirm both tracks exist and are non-empty before you download them.
# Adjust the ChIP-exo filename to whatever you named your makegff.py output.
# Note: this notebook's shell cwd is notebooks/, so repo-root paths need a ../ prefix.

echo "== Annotation track =="
ls -lh ../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
head -2 ../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff

echo
echo "== ChIP-exo track (edit the filename if yours differs) =="
CHIP_GFF=chipexo.gff
ls -lh "$CHIP_GFF" && head -2 "$CHIP_GFF" && echo "rows: $(wc -l < "$CHIP_GFF")"


== Annotation track ==
-rw-rw-rw- 1 vscode root 674K Jul 13 06:03 ../data/reference/ec_annotation_20100903_DHK_cSRNA_with_ortho.gff
NC_000913	Ecocyc_14.1	gene	190	255	.	+	.	locus_tag=b0001;gene=thrL;feature=gene;product=thr operon leader peptide;KpOrtho=KPN_00001;color=00FF00
NC_000913	Ecocyc_14.1	gene	337	2799	.	+	.	locus_tag=b0002;gene=thrA;feature=gene;product=aspartate kinase / homoserine dehydrogenase;KpOrtho=KPN_00002;color=00FF00

== ChIP-exo track (edit the filename if yours differs) ==
-rw-rw-rw- 1 vscode vscode 106M Jul 16 06:15 chipexo.gff
NC_000913	makegff	fiveprime	11	11	1	+	.	depth=1
NC_000913	makegff	fiveprime	13	13	2	+	.	depth=3
rows: 1899476


### Exercise 10 — Load, Locate, and Identify

Open MetaScope on your machine and:

1. **Open both GFFs** (File → Open, or `Ctrl+O` / `Cmd+O`): the annotation *and* your ChIP-exo 5′-end pile-up track.
2. **Find a binding site.** Look for a location where the ChIP-exo track shows a clear, tall **peak** of read counts. Use `Ctrl/Cmd+G` to jump to a position and scroll/zoom to explore. (Iron-uptake regions worth checking: **160–170 kb** near *fhuA*, **609–613 kb** at the *fepA*/*fes* pair, **624–629 kb** at the enterobactin *entCEBA* genes, **1,733 kb** near *sodB*.) If you kept strands separate, the true crosslink sits **between** the flanking `+` and `−` peaks.
3. **Identify the nearest gene.** Hover over a gene feature for its tooltip (position, strand, name), or `Ctrl/Cmd+F` to search the annotation by name. Read off which gene the peak sits **upstream of**.

> **Heads-up — a real gotcha.** MetaScope groups tracks by their **chromosome ID** (column 1 of the GFF). The annotation uses `NC_000913`, but `makegff.py` writes `NC_000913.3` (the accession of the reference FASTA). If your two tracks land on **separate chromosome tabs** and won't line up, that's why. Use `/debug` to reason it out, then fix it (make the two column-1 values match) and reload. Worth verifying rather than assuming.

4. **Export your figure** (`Ctrl+Shift+E` → PNG, 300 dpi) as `module3_chipexo_metascope.png`. This is your Module 3 deliverable.

In the answer cell, record: the genomic position of the site, the nearest gene, and whether the binding site is upstream of it (as you predicted in Exercise 8).

### Exercise 11 — Cross-Check the Biology

You found a gene next to a FUR binding site. Now confirm it makes biological sense — don't take the browser (or Claude) at face value.

Ask Claude Code: *is this gene a known FUR target, and what does it do?* A correct result should be an **iron-acquisition or iron-storage gene** (siderophore transport, enterobactin biosynthesis, iron-superoxide dismutase, etc.) — FUR represses these under iron-replete conditions.

If the gene has nothing to do with iron, that's a signal to re-check: did you pick the right peak? Is it really the *nearest* gene? Are the two tracks on the same chromosome tab? Write what you found.

> **Answer**
>
>
> : fes는 enterochelin esterase로, 철 흡수 대사에 관여하는 효소 유전자이다. Enterobactin이 세포 밖에서 Fe3+를 chelate해서 세포 안으로 들여오고, Fes가  ferric-enterobactin 복합체를 가수분해해서 철을 방출시킨다. fes는 FUR에 의해 조절되는 유전자 중 하나로, 철이 충분할 때, Fur-Fe 복합체가 프로모터의 Fur box에 결합해 전사를 억제하고, 철이 부족하면 억제가 풀려 발현된다.

## End of Session

Before closing:

1. Make sure your pipeline cell runs end-to-end without errors.
2. Verify that the output BAM is coordinate-sorted: `samtools view -H output.bam | grep SO`
3. Verify the index exists: `ls -lh output.bam.bai`
4. Verify your GFF output has content: `wc -l output.gff`
> -> 저장 경로에 맞춰 'wc -l notebooks/chipexo.gff'로 확인하였습니다.
5. In MetaScope, load your GFF (Section 8) and export the figure `module3_chipexo_metascope.png`.
6. Run `/log` in Claude Code to save a session log.

---

**Run `/log` before closing.**

## Git — Commit Your Work

Every session ends with a commit. Run the commands below in your terminal (not in this notebook).

Write your own commit message following the format: `feat(module3): <what you did>`.  
If you're unsure what to write, ask Claude Code to suggest one based on what you worked on.

In [ ]:
# Run these in the terminal (not here):
# git add notebooks/
# git commit -m "..."   # write your own commit message; use Claude Code to help if needed
# git push
